In [ ]:
import os
import zarr
import s3fs
import xarray as xr
from rechunker import rechunk
import dask
import dask.array as da
from dask import compute
from dask_jobqueue import SLURMCluster
from dask.distributed import Client, progress

# For connecting to dashboard

In [ ]:
dask.config.config.get('distributed').get('dashboard').update({'link':'{JUPYTERHUB_SERVICE_PREFIX}/proxy/{port}/status'})

# Initialization and store paths etc. 

In [ ]:
storeName           = "ngc4008_PT15M_9.zarr"  # 👈 Change path accordingly
s3preProdURL        = "https://s3.eu-dkrz-3.dkrz.cloud" # 👈 Change path accordingly
s3preProdAccesskey  = os.getenv( "ACCESS_KEY_S3_nextGEMS" ) # 👈 Change path accordingly
s3preProdSecretkey  = os.getenv( "SECRET_KEY_S3_nextGEMS" ) # 👈 Change path accordingly
target_storePath    = f'nextgems/timeChunks/{storeName}' 

storage_options = {
    "key": s3preProdAccesskey,
    "secret": s3preProdSecretkey,
    "client_kwargs": {"endpoint_url": s3preProdURL},
    "config_kwargs": {"max_pool_connections": 150}, # parallel reads  # 👈 tune this as per the S3 setup
}

targetS3Cluster = s3fs.S3FileSystem( **storage_options )

# Local directory source store

In [ ]:
currentSrcStore       = f"/work/bk1414/k204247/S3toHSM/daskExercise/rechunking/nextgemsSingleTimestep/{storeName}"
src = zarr.open_consolidated( currentSrcStore, mode="r", zarr_version = 2 ) # Local irectory store
print( src.tree() )

In [ ]:
dstStore = s3fs.S3Map( target_storePath, s3 = targetS3Cluster, check = False ) # Remote S3
dst = zarr.open_group( dstStore, mode="w", zarr_version = 2 )

# Allocate the resources using slurm cluster

In [ ]:
cluster = SLURMCluster(name='dask-cluster',
                      cores=1,
                      memory='16GB',# 👈 Can be tuned
                      processes=1,
                      interface='ib0',
                      queue='compute',
                      account='bk1414',
                      walltime='08:00:00', # 👈 Max. time limit on 'compute'
                      asynchronous=0)
cluster.scale(jobs=15)  # 👈 number of SLURM jobs
client = Client(cluster)

In [ ]:
client

In [ ]:
def copy_array(src_array, dst_group, name):
    
    d = da.from_zarr(src_array)
    
    return d.to_zarr(
        dst_group.require_dataset(
            name,
            shape=src_array.shape,
            chunks=d.chunksize,
            dtype=src_array.dtype,
        ),
        storage_options=storage_options,
        overwrite=False,
        compute=False,  # IMPORTANT for task graph
    )


def copy_group(src_group, dst_group):
    
    tasks = []
    
    for name, array in src_group.arrays():
        tasks.append(copy_array(array, dst_group, name))
    
    for name, subgroup in src_group.groups():
        dst_sub = dst_group.require_group(name)
        tasks.extend(copy_group(subgroup, dst_sub))
    
    return tasks

In [ ]:
tasks = copy_group(src, dst)

In [ ]:
with dask.config.set( scheduler = "distributed" ):
    dask.compute(*tasks)

# Consolidate metadata

In [ ]:
zarr.consolidate_metadata( dstStore )

In [ ]:
client.close()

In [ ]:
client.status

# Verify target store

In [ ]:
target_group = zarr.open( dstStore, mode='r' )
print( f"{ [ target_group[arr[0]].info for arr in target_group.arrays() ] }\n" )
print( f"{ xr.open_dataset(dstStore) }\n" )